# D393 - Snowflake external tables, complete practical guide

An external table gives Snowflake a read-only relational interface over files that remain in cloud object storage. This notebook covers stages, file formats, the `VALUE` column, virtual columns, nested JSON, file metadata, refresh modes, computed and user-specified partitions, schema drift, materialization, streams, performance, security, monitoring, and cleanup.

Apache Iceberg is intentionally absent from this notebook. Iceberg is a table format with snapshots and transactional metadata; it is taught completely in D398.

## 1. External stage, external table, and external volume

| Object | Purpose | Owns data? |
|---|---|---|
| External stage | Named connection to a bucket/container path, credentials, and optional file format | No |
| File format | Parsing rules for CSV, JSON, Parquet, Avro, or ORC | No |
| External table | Read-only table metadata over files registered from an external stage | No |
| External volume | Storage locations and IAM trust for Apache Iceberg tables | No; unrelated to ordinary external tables |

Every external-table row is exposed through a `VALUE` column of type `VARIANT`. `METADATA$FILENAME` and `METADATA$FILE_ROW_NUMBER` identify its source. Virtual columns cast paths from `VALUE` or derive values from the filename. Snowflake stores the definition and registered file list; the bytes remain in the stage location.

External tables support `SELECT`, views, joins, and streams that track file-registration additions. They do not support row-level `INSERT`, `UPDATE`, `DELETE`, `MERGE`, or `TRUNCATE`. Modify files in the owning system and refresh metadata, or copy rows into a writable Snowflake/Iceberg table. Source: [external-table introduction](https://docs.snowflake.com/en/user-guide/tables-external-intro).

## 2. Create the lab environment

The runnable example uses public JSON maintained for Snowflake's official tutorial, so it requires no personal cloud credentials. Organizational network policies can still block public storage; if that happens, replace the stage with an approved stage.

```sql
USE ROLE SYSADMIN;
CREATE DATABASE IF NOT EXISTS D39_TABLE_LAB;
CREATE SCHEMA IF NOT EXISTS D39_TABLE_LAB.EXTERNAL_DATA;
CREATE WAREHOUSE IF NOT EXISTS D39_LAB_WH
  WAREHOUSE_SIZE = 'X-SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE D39_LAB_WH;
USE DATABASE D39_TABLE_LAB;
USE SCHEMA EXTERNAL_DATA;
```

## 3. Define parsing separately from location

A named file format centralizes parsing behavior. A stage points to the location. `LIST` is the first diagnostic: if it cannot see files, the table cannot register them. Production private locations should use a storage integration rather than embedding long-lived keys in SQL.

```sql
CREATE OR REPLACE FILE FORMAT EXT_JSON_FORMAT
  TYPE = JSON
  STRIP_OUTER_ARRAY = TRUE;

CREATE OR REPLACE STAGE EXT_PUBLIC_JSON_STAGE
  URL = 's3://snowflake-docs/tutorials/json'
  FILE_FORMAT = EXT_JSON_FORMAT;

LIST @EXT_PUBLIC_JSON_STAGE;
```

Common alternatives:

```sql
CREATE OR REPLACE FILE FORMAT EXT_CSV_FORMAT
  TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  EMPTY_FIELD_AS_NULL = TRUE NULL_IF = ('NULL', 'null', '');

CREATE OR REPLACE FILE FORMAT EXT_PARQUET_FORMAT TYPE = PARQUET;
```

## 4. Create an external table and inspect raw rows

The required `VALUE` column exists automatically. `SELECT *` returns it; metadata pseudocolumns must be selected explicitly. `AUTO_REFRESH = FALSE` keeps this lab deterministic and avoids requiring cloud event notification configuration.

```sql
CREATE OR REPLACE EXTERNAL TABLE APP_EVENTS_EXTERNAL
WITH LOCATION = @EXT_PUBLIC_JSON_STAGE
AUTO_REFRESH = FALSE
REFRESH_ON_CREATE = TRUE
FILE_FORMAT = (FORMAT_NAME = EXT_JSON_FORMAT)
COMMENT = 'Public JSON external-table lab';

SELECT VALUE, TYPEOF(VALUE) AS VALUE_TYPE,
       METADATA$FILENAME AS SOURCE_FILE,
       METADATA$FILE_ROW_NUMBER AS FILE_ROW_NUMBER
FROM APP_EVENTS_EXTERNAL
ORDER BY SOURCE_FILE, FILE_ROW_NUMBER
LIMIT 20;
```

Creation performs one metadata refresh because `REFRESH_ON_CREATE = TRUE`. On paths with millions of files, create with it disabled and refresh manageable subpaths incrementally.

## 5. Add typed virtual columns

Virtual columns are expressions evaluated over external rows. `TRY_TO_*` conversions are safer during discovery because malformed values become null instead of failing the query. After profiling and enforcing producer contracts, stricter casts can expose bad data immediately.

```sql
CREATE OR REPLACE EXTERNAL TABLE APP_EVENTS_EXTERNAL (
  DEVICE_TYPE VARCHAR AS (VALUE:device_type::VARCHAR),
  EVENTS ARRAY AS (VALUE:events::ARRAY),
  SOURCE_FILE VARCHAR AS (METADATA$FILENAME),
  SOURCE_ROW NUMBER AS (METADATA$FILE_ROW_NUMBER)
)
WITH LOCATION = @EXT_PUBLIC_JSON_STAGE
AUTO_REFRESH = FALSE
FILE_FORMAT = (FORMAT_NAME = EXT_JSON_FORMAT);

SELECT DEVICE_TYPE, ARRAY_SIZE(EVENTS) AS EVENT_COUNT, SOURCE_FILE, SOURCE_ROW
FROM APP_EVENTS_EXTERNAL
ORDER BY SOURCE_FILE, SOURCE_ROW
LIMIT 20;

DESCRIBE EXTERNAL TABLE APP_EVENTS_EXTERNAL;
```

## 6. Query arrays and nested objects

`LATERAL FLATTEN` emits one row per array element. Keep filename and row number during ingestion so rejected or surprising values can be traced to an exact file record.

```sql
SELECT E.DEVICE_TYPE, E.SOURCE_FILE, E.SOURCE_ROW,
       F.INDEX AS EVENT_INDEX,
       TRY_TO_NUMBER(F.VALUE:f) AS EVENT_CODE,
       TO_TIMESTAMP_NTZ(TRY_TO_NUMBER(F.VALUE:t) / 1000) AS EVENT_TS,
       F.VALUE:v AS MEASUREMENTS
FROM APP_EVENTS_EXTERNAL E,
LATERAL FLATTEN(INPUT => E.EVENTS) F
ORDER BY E.SOURCE_FILE, E.SOURCE_ROW, EVENT_INDEX
LIMIT 50;

SELECT DEVICE_TYPE, COUNT(*) AS SOURCE_ROWS,
       SUM(ARRAY_SIZE(EVENTS)) AS NESTED_EVENTS
FROM APP_EVENTS_EXTERNAL
GROUP BY DEVICE_TYPE
ORDER BY DEVICE_TYPE;
```

## 7. File registration and refresh lifecycle

An external table queries registered files. Adding, replacing, or deleting a cloud object does not reliably change results until Snowflake refreshes table metadata. Refresh can cover the full location, a relative subpath, or explicit file additions/removals. These operations change Snowflake metadata; they never edit cloud files.

```sql
ALTER EXTERNAL TABLE APP_EVENTS_EXTERNAL REFRESH;

-- Production examples using paths relative to LOCATION:
-- ALTER EXTERNAL TABLE APP_EVENTS_EXTERNAL REFRESH 'year=2026/month=09/';
-- ALTER EXTERNAL TABLE APP_EVENTS_EXTERNAL ADD FILES ('year=2026/month=09/orders-01.json');
-- ALTER EXTERNAL TABLE APP_EVENTS_EXTERNAL REMOVE FILES ('year=2026/month=08/orders-old.json');

SHOW EXTERNAL TABLES LIKE 'APP_EVENTS_EXTERNAL';
SELECT *
FROM TABLE(D39_TABLE_LAB.INFORMATION_SCHEMA.EXTERNAL_TABLE_FILES(
  TABLE_NAME => 'D39_TABLE_LAB.EXTERNAL_DATA.APP_EVENTS_EXTERNAL'
))
ORDER BY FILE_NAME;
```

A file replacement at the same path should be treated carefully because notification ordering and caches can complicate operational reasoning. Immutable file names plus add/remove publication are easier to audit. Source: [ALTER EXTERNAL TABLE](https://docs.snowflake.com/en/sql-reference/sql/alter-external-table).

## 8. Automatic refresh is a cloud event pipeline

`AUTO_REFRESH = TRUE` alone is insufficient. Configure the provider's event system and a Snowflake notification integration so object-created and object-removed events reach Snowflake. S3 commonly uses SQS/SNS, Azure uses Event Grid, and GCS uses Pub/Sub. Run one manual refresh after setup to close the gap between table creation and notification activation.

```sql
-- Run only after the cloud notification integration is configured.
-- ALTER EXTERNAL TABLE APP_EVENTS_EXTERNAL SET AUTO_REFRESH = TRUE;
-- ALTER EXTERNAL TABLE APP_EVENTS_EXTERNAL REFRESH;
-- SELECT SYSTEM$EXTERNAL_TABLE_PIPE_STATUS(
--   'D39_TABLE_LAB.EXTERNAL_DATA.APP_EVENTS_EXTERNAL'
-- ) AS REFRESH_STATUS;
```

Ownership transfer disables automatic refresh by default. Add re-enabling and status verification to the ownership-change runbook. User-specified partitions and S3-compatible stages do not support automatic refresh. Source: [automatic external-table refresh](https://docs.snowflake.com/en/user-guide/tables-external-auto).

## 9. Computed partitions from file paths

Partition columns prune whole files before row expressions are evaluated. Organize paths around stable, selective dimensions such as `order_date` or `region`, then filter on those partition columns. This production template expects paths like `orders/year=2026/month=09/day=07/file.parquet`.

```sql
-- CREATE OR REPLACE STAGE ECOMMERCE_PARQUET_STAGE
--   URL = 's3://your-bucket/ecommerce/'
--   STORAGE_INTEGRATION = YOUR_STORAGE_INTEGRATION
--   FILE_FORMAT = EXT_PARQUET_FORMAT;
--
-- CREATE OR REPLACE EXTERNAL TABLE ORDERS_EXTERNAL (
--   ORDER_YEAR NUMBER AS TRY_TO_NUMBER(REGEXP_SUBSTR(METADATA$FILENAME, 'year=([0-9]{4})', 1, 1, 'e', 1)),
--   ORDER_MONTH NUMBER AS TRY_TO_NUMBER(REGEXP_SUBSTR(METADATA$FILENAME, 'month=([0-9]{2})', 1, 1, 'e', 1)),
--   ORDER_ID NUMBER AS VALUE:order_id::NUMBER,
--   CUSTOMER_ID NUMBER AS VALUE:customer_id::NUMBER,
--   ORDER_TS TIMESTAMP_NTZ AS VALUE:order_ts::TIMESTAMP_NTZ,
--   STATUS VARCHAR AS VALUE:status::VARCHAR,
--   ORDER_TOTAL NUMBER(12,2) AS VALUE:order_total::NUMBER(12,2)
-- )
-- PARTITION BY (ORDER_YEAR, ORDER_MONTH)
-- LOCATION = @ECOMMERCE_PARQUET_STAGE/orders/
-- AUTO_REFRESH = FALSE
-- FILE_FORMAT = (FORMAT_NAME = EXT_PARQUET_FORMAT);
--
-- SELECT ORDER_ID, STATUS, ORDER_TOTAL
-- FROM ORDERS_EXTERNAL
-- WHERE ORDER_YEAR = 2026 AND ORDER_MONTH = 9;
```

Partitioning narrows files, not individual rows inside each selected file. Avoid high-cardinality path dimensions that produce many tiny files.

## 10. User-specified partitions

Use manual partitions when another metastore or governance process decides exactly which locations are visible. This mode cannot use ordinary manual/automatic refresh; the owner adds and drops partitions. The mode cannot be changed after creation, so choose it deliberately.

```sql
-- CREATE OR REPLACE EXTERNAL TABLE ORDERS_EXTERNAL_MANUAL (
--   ORDER_DATE DATE AS PARSE_JSON(METADATA$EXTERNAL_TABLE_PARTITION):ORDER_DATE::DATE,
--   REGION VARCHAR AS PARSE_JSON(METADATA$EXTERNAL_TABLE_PARTITION):REGION::VARCHAR,
--   ORDER_ID NUMBER AS VALUE:order_id::NUMBER,
--   ORDER_TOTAL NUMBER(12,2) AS VALUE:order_total::NUMBER(12,2)
-- )
-- PARTITION BY (ORDER_DATE, REGION)
-- LOCATION = @ECOMMERCE_PARQUET_STAGE/orders/
-- PARTITION_TYPE = USER_SPECIFIED
-- FILE_FORMAT = (FORMAT_NAME = EXT_PARQUET_FORMAT);
--
-- ALTER EXTERNAL TABLE ORDERS_EXTERNAL_MANUAL
--   ADD PARTITION (ORDER_DATE='2026-09-07', REGION='SOUTH')
--   LOCATION 'order_date=2026-09-07/region=SOUTH/';
--
-- ALTER EXTERNAL TABLE ORDERS_EXTERNAL_MANUAL
--   DROP PARTITION LOCATION 'order_date=2026-09-07/region=SOUTH/';
```

## 11. Profile and quarantine schema drift

External-table virtual columns do not evolve the source files. Profile raw `VALUE`, classify records, then materialize valid rows. Keep source metadata in the quarantine result.

```sql
SELECT OBJECT_KEYS(VALUE) AS TOP_LEVEL_KEYS, COUNT(*) AS ROWS
FROM APP_EVENTS_EXTERNAL
GROUP BY TOP_LEVEL_KEYS;

SELECT SOURCE_FILE, SOURCE_ROW, VALUE
FROM APP_EVENTS_EXTERNAL
WHERE DEVICE_TYPE IS NULL OR NOT IS_ARRAY(EVENTS);

CREATE OR REPLACE TRANSIENT TABLE APP_EVENTS_QUARANTINE AS
SELECT SOURCE_FILE, SOURCE_ROW, VALUE, CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM APP_EVENTS_EXTERNAL
WHERE DEVICE_TYPE IS NULL OR NOT IS_ARRAY(EVENTS);
```

For Parquet, Avro, and ORC, physical schemas also matter. A conversion error while scanning can cause a file to be skipped and a query can return rows read before the error, so monitoring and producer contracts are essential.

## 12. Materialize external data when rows must change

Copying into a native table creates a writable, optimized Snowflake representation. Add ingestion metadata for traceability and a file/row key for deduplication.

```sql
CREATE OR REPLACE TABLE APP_EVENTS_NATIVE AS
SELECT DEVICE_TYPE, EVENTS, SOURCE_FILE, SOURCE_ROW, CURRENT_TIMESTAMP() AS INGESTED_AT
FROM APP_EVENTS_EXTERNAL;

UPDATE APP_EVENTS_NATIVE
SET DEVICE_TYPE = UPPER(DEVICE_TYPE);
DELETE FROM APP_EVENTS_NATIVE WHERE DEVICE_TYPE IS NULL;

SELECT DEVICE_TYPE, COUNT(*) AS ROWS
FROM APP_EVENTS_NATIVE GROUP BY DEVICE_TYPE ORDER BY DEVICE_TYPE;
```

For repeatable loads, use a stream or a file-ledger table and `MERGE` on a stable event key. CTAS above intentionally shows the simplest snapshot rather than an incremental production pipeline.

## 13. Streams track newly registered external files

An external-table stream is insert-only. It reports rows from newly registered files; it is not row-level CDC for modifications inside previously registered files. Create the stream before the next metadata refresh.

```sql
CREATE OR REPLACE STREAM APP_EVENTS_EXTERNAL_STREAM
  ON EXTERNAL TABLE APP_EVENTS_EXTERNAL INSERT_ONLY = TRUE;

ALTER EXTERNAL TABLE APP_EVENTS_EXTERNAL REFRESH;

SELECT DEVICE_TYPE, SOURCE_FILE, SOURCE_ROW, METADATA$ACTION
FROM APP_EVENTS_EXTERNAL_STREAM
ORDER BY SOURCE_FILE, SOURCE_ROW;

-- A DML consumer advances the offset only when its transaction commits.
INSERT INTO APP_EVENTS_NATIVE (DEVICE_TYPE, EVENTS, SOURCE_FILE, SOURCE_ROW, INGESTED_AT)
SELECT DEVICE_TYPE, EVENTS, SOURCE_FILE, SOURCE_ROW, CURRENT_TIMESTAMP()
FROM APP_EVENTS_EXTERNAL_STREAM;
```

If no files were added after stream creation, the result is empty. Replacing a file is not a safe row-change protocol; publish immutable new file names.

## 14. Performance and cost controls

1. Partition paths and filter on partition columns so Snowflake prunes files.
2. Prefer Parquet with complete statistics for analytical data. Snowflake recommends roughly 256-512 MB Parquet files and 16-256 MB row groups; other formats commonly use 16-256 MB files.
3. Avoid millions of tiny files; listing, metadata refresh, planning, and object-store requests become expensive.
4. Select only needed virtual columns and predicates.
5. Materialize reused transformations into native or Iceberg tables.
6. Enterprise Edition can use a materialized view over an external table; refresh external metadata so the view sees the current file set.
7. Use Query Profile to separate remote scan time, bytes scanned, pruning, and warehouse work.

```sql
-- Enterprise Edition pattern; validate materialized-view restrictions first.
-- CREATE MATERIALIZED VIEW APP_EVENT_COUNTS_MV AS
-- SELECT DEVICE_TYPE, COUNT(*) AS EVENT_ROWS
-- FROM APP_EVENTS_EXTERNAL
-- GROUP BY DEVICE_TYPE;
```

## 15. Security and operating checklist

- Give the storage integration the narrowest cloud path and read permissions required.
- Separate stage ownership, external-table ownership, and consumer `SELECT` grants.
- Do not place secrets in stage URLs, comments, or worksheet text.
- Treat filenames and path partitions as potentially sensitive metadata.
- Monitor refresh failures, notification lag, registered-file counts, and unexpected schema/null rates.
- Test removal behavior before deleting cloud files; a registered path is not a backup.
- Keep cloud versioning and retention aligned with recovery needs because external-table Time Travel cannot restore deleted source bytes.
- Re-enable and verify `AUTO_REFRESH` after ownership transfer.

```sql
SHOW EXTERNAL TABLES LIKE 'APP_EVENTS_EXTERNAL';
DESCRIBE EXTERNAL TABLE APP_EVENTS_EXTERNAL;
SHOW STAGES LIKE 'EXT_PUBLIC_JSON_STAGE';
SHOW FILE FORMATS LIKE 'EXT_JSON_FORMAT';
```

## 16. Optional cleanup

```sql
USE SCHEMA D39_TABLE_LAB.EXTERNAL_DATA;
DROP STREAM IF EXISTS APP_EVENTS_EXTERNAL_STREAM;
DROP TABLE IF EXISTS APP_EVENTS_NATIVE;
DROP TABLE IF EXISTS APP_EVENTS_QUARANTINE;
DROP EXTERNAL TABLE IF EXISTS APP_EVENTS_EXTERNAL;
DROP STAGE IF EXISTS EXT_PUBLIC_JSON_STAGE;
DROP FILE FORMAT IF EXISTS EXT_PARQUET_FORMAT;
DROP FILE FORMAT IF EXISTS EXT_CSV_FORMAT;
DROP FILE FORMAT IF EXISTS EXT_JSON_FORMAT;
```

Keep the shared database and warehouse for the remaining D39 notebooks.